# Permutation test 
Verifica la correlazione della cca fatta sul train

## Parametri

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from source.cca import load_tensor, fit, pearson_corr

tensors_dir = project_root / "data" / "results" / "tensors"
results_dir = project_root / "data" / "results" / "cca"

arrangement_types = ["bass", "full", "full_no_bass", "harm", "lead"]

seed = 6
n_permutation = 5000

## Permutazione delle batterie tra tutte le tracce

In [2]:
def permutation_test(X, Y, n_permutation, seed):

    rng = np.random.default_rng(seed)
    obs = pearson_corr(fit(X, Y), X, Y) # valore della correlazione per arrangiamento e batteria originali
    null = np.empty(n_permutation) # valore della correlazione se non c'è relazione tra arrangiamento e batteria

    for i in range(n_permutation):
        Yp = Y[rng.permutation(len(Y))] # batteria casuale
        null[i] = pearson_corr(fit(X, Yp), X, Yp) # correlazione tra arrangiamento e batteria casuale

    p_value = (1 + np.sum(null >= obs)) / (n_permutation + 1) # probabilità di ottenere una correlazione data da arrangiamento e batteria casuale >= a quella osservata)
    return obs, null, p_value

righe = []
for arrangement_type in arrangement_types:
    tensor = load_tensor(tensors_dir / f"tensor_train_{arrangement_type}.npz")
    obs, null, p_value = permutation_test(tensor["X"], tensor["Y"], n_permutation, seed)
    righe.append(dict(classe=arrangement_type,
                      n=len(tensor["ids"]),
                      observation=obs,
                      null_media=null.mean(),
                      null_p95=np.percentile(null, 95),
                      p_value=p_value,
                      n_permutation=n_permutation,
                      seed=seed
                      ))

permutationTest = pd.DataFrame(righe)

# controllo della significatività dei p-value con correzione di Holm
reject, p_holm, _, _ = multipletests(permutationTest["p_value"], method="holm")
permutationTest["p_holm"] = p_holm
permutationTest["reject_holm"] = reject
permutationTest.to_parquet(results_dir / "permutation_test.parquet")
permutationTest

,classe,n,observation,null_media,null_p95,p_value,n_permutation,seed,p_holm,reject_holm
0,bass,32,0.405649,0.259906,0.307049,0.000200,5000,6,0.001000,True
1,full,43,0.390972,0.317090,0.341245,0.000200,5000,6,0.001000,True
2,full_no_bass,54,0.388119,0.344932,0.365703,0.000600,5000,6,0.001200,True
3,harm,51,0.385519,0.324207,0.348170,0.000200,5000,6,0.001000,True
4,lead,44,0.225791,0.208319,0.227001,0.062587,5000,6,0.062587,False
